## Goal
Build a canonical resume from an existing resume in JSON. Validate conversion process by converting back into a resume.
- Stage 1 was developing the Master Resume doc which covered entire career history.

In [ ]:
from pathlib import Path
import json
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
from src.config import SOURCE_DIR, LAYOUT_DIR, CONTRACT_DIR, ARTIFACT_DIR
from src.helpers import (
    extract_docx_text,
    load_json,
    parse_json_response,
    save_json
)
from src.renderer import (
    load_yaml,
    render_resume,
    convert_docx_to_pdf,
)

ARTIFACT_DIR.mkdir(exist_ok=True)


master_docx_path = SOURCE_DIR / "Master Resume.docx"
schema_path = CONTRACT_DIR / "Canonical Resume Schema.md"
canonical_master_resume_path = ARTIFACT_DIR / "canonical_master_resume.json"

load_dotenv()
client = OpenAI()
MODEL = "gpt-4.1-mini"



In [ ]:
master_resume_text = extract_docx_text(master_docx_path)
schema_text = schema_path.read_text(encoding="utf-8")

In [3]:
def build_canonical_resume_prompt(master_resume_text, schema_text):
    return f"""
Convert the provided master resume into canonical resume JSON.

Follow the canonical resume schema exactly.

Important rules:
- Preserve career evidence.
- Do not shorten accomplishments.
- Do not optimize for a target role.
- Do not add facts not present in the master resume.
- Put resume content into structured sections.
- Put non-resume background material into an Archive section if needed.
- Return valid JSON only.

Canonical Resume Schema:
{schema_text}

Master Resume:
{master_resume_text}
"""

In [4]:
prompt = build_canonical_resume_prompt(master_resume_text, schema_text)

In [6]:
response = client.responses.create(
    model=MODEL,
    input=prompt,
    temperature=0,
    text={
        "format": {"type": "json_object"}
    },
)

canonical_resume = parse_json_response(response)

In [7]:
# Basic validation
def validate_canonical_resume(resume):
    assert "header" in resume
    assert "sections" in resume
    assert isinstance(resume["sections"], list)

    for section in resume["sections"]:
        assert "heading" in section
        assert "type" in section
        assert "content" in section

    return True

validate_canonical_resume(canonical_resume)

True

In [ ]:
save_json(
    canonical_resume,
    ARTIFACT_DIR / "canonical_master_resume.json"
)

17611

In [ ]:

target_resume = load_json(canonical_master_resume_path)
layout = load_yaml(LAYOUT_DIR / "standard_v6.yaml")
output_path = ARTIFACT_DIR / "resume_v6.docx"

render_resume(
    target_resume, layout, output_path)

convert_docx_to_pdf(output_path)

convert /Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks/resume-builder/artifacts/resume_v6.docx as a Writer document -> /Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks/resume-builder/artifacts/resume_v6.pdf using filter : writer_pdf_Export


PosixPath('/Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks/resume-builder/artifacts/resume_v6.pdf')

# Phase 2 Summary: Canonical Resume Schema & Rendering

## Objective

Define a stable, renderer-independent representation of resume content that can serve as the foundation for all future resume generation activities.

## Artifacts Produced

### Contracts

* Canonical Resume Schema.md

### Data

* canonical_master_resume.json

### Rendering

* renderer.py
* standard.yaml
* canonical_master_resume.docx
* canonical_master_resume.pdf

## Key Design Decisions

### Separate Content from Presentation

Resume content is stored in canonical JSON.

Presentation concerns such as fonts, spacing, alignment, bolding, and page layout are defined in YAML layout files and renderer logic.

### Keep the Schema Small

The schema intentionally uses a small number of content types:

* paragraph
* bullet
* inline_list
* subsections
* experience

Specialized structures such as education, certifications, awards, patents, and technology categories are represented using existing schema elements rather than introducing new content types.

### Use Subsections as the Primary Reusable Structure

The subsection object became the primary mechanism for representing labeled child records.

Examples include:

* Core Technologies
* Core Expertise
* Education
* Patents
* Awards
* Certifications
* Client engagements within consulting roles

### Renderer Independence

The canonical resume contains semantic information only.

Renderers determine visual presentation.

The same canonical JSON can be rendered using multiple layouts without modifying the underlying content.

## Validation Results

The canonical resume was successfully rendered to both DOCX and PDF formats.

Schema issues discovered during rendering led to several improvements, including:

* replacing flattened education bullets with structured subsections
* simplifying content types
* strengthening subsection semantics
* removing presentation concerns from content artifacts

The successful round-trip from:

Master Resume
→ Canonical Resume JSON
→ DOCX/PDF

demonstrates that the schema is sufficiently expressive for resume generation.

## Lessons Learned

The most important lesson from this phase was the importance of defining contracts before generation.

The renderer exposed weaknesses in the schema much earlier than downstream resume-generation workflows would have.

By validating the schema through rendering, structural issues were discovered and resolved before building archetypes, capability extraction, scoring pipelines, and resume-generation workflows.

## Exit Criteria

Phase 2 is considered complete when:

* the schema is documented
* the canonical resume can be generated
* the canonical resume can be rendered successfully
* layout changes can be made without modifying resume content

These criteria have been satisfied.

## Next Phase

Phase 3 focuses on market discovery.

Inputs:

* canonical_master_resume.json
* job descriptions

Output:

* target_archetype.json

The objective is to identify the capabilities, themes, priorities, and evidence patterns most valued by the target market before evaluating resume content.
